1. Look
2. Find the problems
3. Fix them one at a time
4. Never change the original
5. Write the steps as code


In [1]:
import pandas as pd

orders_raw = pd.read_csv('data/orders.csv')
orders = orders_raw.copy()

orders.shape

(825, 10)

## First look

In [2]:
orders.head(8)

,order_id,customer_id,order_date,product,category,unit_price,quantity,express,coupon_code,gift_message
0,ORD-20001,C0259,2026-01-05,Rain jacket,Clothing,"1 199,00 kr",2,No,WELCOME10,NaN
1,ORD-20002,C0210,2026-01-05,Sandals,Shoes,349 kr,2,No,FRIEND15,NaN
2,ORD-20003,C0238,2026-01-05,Trail running shoes,Shoes,"1 039,20 kr",1,yes,NaN,NaN
3,ORD-20004,C0252,2026-01-05,Cookbook,Books,249 kr,1,no,NaN,NaN
4,ORD-20005,C0179,2026-01-05,Trail running shoes,Shoes,1299 kr,1,no,WELCOME10,NaN
5,ORD-20006,C0143,2026-01-05,Garden hose,Home & Garden,349 kr,2,NO,NaN,NaN
6,ORD-20007,C0293,2026-01-05,Phone charger,Electronics,199 kr,2,Yes,NaN,NaN
7,ORD-20008,C0184,2026-01-05,Trail running shoes,Shoes,"1 039,20 kr",2,No,NaN,NaN


In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 825 entries, 0 to 824
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      825 non-null    str    
 1   customer_id   825 non-null    str    
 2   order_date    824 non-null    str    
 3   product       825 non-null    str    
 4   category      825 non-null    str    
 5   unit_price    825 non-null    str    
 6   quantity      825 non-null    int64  
 7   express       825 non-null    str    
 8   coupon_code   170 non-null    str    
 9   gift_message  0 non-null      float64
dtypes: float64(1), int64(1), str(8)
memory usage: 64.6 KB


## Text I: spaces and capital letters

In [4]:
orders['category'].nunique()

38

In [5]:
orders['category'].value_counts()

category
Clothing            128
Shoes               101
Sports               99
Electronics          97
Home & Garden        82
Books                55
ELECTRONICS          18
clothing             18
electronics          17
SPORTS               16
sports               16
Home and Garden      16
Clothing             14
CLOTHING             14
SHOES                13
HOME & GARDEN        11
 Electronics         10
Shoes                10
home & garden        10
books                 9
shoes                 9
Sports                9
 Shoes                9
Electronics           9
 Sports               6
 Home & Garden        6
 Clothing             6
Books                 4
Home & Garden         4
HOME & GARDEN         1
 Books                1
home and garden       1
BOOKS                 1
 Home and Garden      1
Home and Garden       1
 Shoes                1
SPORTS                1
 Sports               1
Name: count, dtype: int64

In [6]:
print((orders['category'] == 'Clothing').sum())

128


.str

In [7]:
print(orders['category'].nunique())
print(orders['category'].str.strip().nunique())
print(orders['category'].str.strip().str.lower().nunique())


38
20
7


In [8]:
orders['category'].str.strip().str.lower().value_counts()

category
clothing           180
electronics        151
sports             148
shoes              143
home & garden      114
books               70
home and garden     19
Name: count, dtype: int64

In [ ]:
orders['category'] = (
    orders['category']
    .str.strip()
    .str.lower()
    .replace('home and garden', 'home & garden')
    .str.title()
)

orders['category'].value_counts()

category
Clothing         180
Electronics      151
Sports           148
Shoes            143
Home & Garden    133
Books             70
Name: count, dtype: int64

In [10]:
orders['product'].nunique()

24

## Text II: yes/no and prices that are text

In [11]:
orders['express'].value_counts()

express
no     300
No     166
yes    152
NO      88
Yes     81
y       38
Name: count, dtype: int64

In [12]:
orders['express'] = orders['express'].str.lower().map({'yes':True, 'y':True, 'no':False})

print(orders['express'].isna().sum())
orders['express'].value_counts()

0


express
False    554
True     271
Name: count, dtype: int64

In [13]:
orders['unit_price'].head(8).tolist()

['1 199,00 kr',
 '349 kr',
 '1 039,20 kr',
 '249 kr',
 '1299 kr',
 '349 kr',
 '199 kr',
 '1 039,20 kr']

In [14]:
orders['unit_price'].astype(float)

ValueError: could not convert string to float: '1 199,00 kr'

In [ ]:
price = orders['unit_price'].str.strip()
print(price.head(3).tolist())

price = price.str.replace(' kr', '')
print(price.head(3).tolist())

price=price.str.replace(' ', '')
print(price.head(3).tolist())

price = price.str.replace(',', '.') 
print(price.head(3).tolist())

# price = orders['unit_price'].str.replace(r'\s|kr', '', regex=True).str.replace(',', '.')


['1 199,00 kr', '349 kr', '1 039,20 kr']
['1 199,00', '349', '1 039,20']
['1199,00', '349', '1039,20']
['1199.00', '349', '1039.20']


In [16]:
orders['unit_price'] = price.astype(float)

print(orders['unit_price'].dtype)
orders['unit_price'].describe().round(2)

float64


count     825.00
mean      429.06
std       374.35
min        59.00
25%       149.00
50%       249.00
75%       699.00
max      1499.00
Name: unit_price, dtype: float64

## Dates: pd.to_datetime()

In [17]:
pd.to_datetime(orders['order_date'])

ValueError: time data "unknown" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [18]:
dates = pd.to_datetime(orders['order_date'], errors='coerce') # becomes NaT

print(orders['order_date'].isna().sum())    # missing before
print(dates.isna().sum())                   # missing after

orders.loc[dates.isna(), ['order_id', 'order_date']]

1
4


,order_id,order_date
51,ORD-20048,unknown
100,ORD-20096,NaN
537,ORD-20523,2026-02-30
784,ORD-20764,2026-04-31


In [19]:
orders['order_date'] = dates

print(orders['order_date'].dtype)
print(orders['order_date'].min())
print(orders['order_date'].max())
print(orders['order_date'].max() - orders['order_date'].min())




datetime64[us]
2026-01-05 00:00:00
2026-06-28 00:00:00
174 days 00:00:00


orders per month using .dt

In [20]:
orders['order_date'].dt.to_period('M').value_counts().sort_index()



order_date
2026-01    136
2026-02    135
2026-03    140
2026-04    130
2026-05    162
2026-06    118
Freq: M, Name: count, dtype: int64

## Duplicates: duplicated()

In [21]:
print(orders.duplicated().sum())

25


In [22]:
orders[orders.duplicated(keep=False)].head(6)

,order_id,customer_id,order_date,product,category,unit_price,quantity,express,coupon_code,gift_message
26,ORD-20027,C0115,2026-01-09,Fleece sweater,Clothing,699.0,1,False,NaN,NaN
27,ORD-20027,C0115,2026-01-09,Fleece sweater,Clothing,699.0,1,False,NaN,NaN
34,ORD-20034,C0285,2026-01-10,Plant pot,Home & Garden,89.0,1,False,NaN,NaN
35,ORD-20034,C0285,2026-01-10,Plant pot,Home & Garden,89.0,1,False,NaN,NaN
40,ORD-20039,C0128,2026-01-10,Bike light,Electronics,199.2,2,True,NaN,NaN
41,ORD-20039,C0128,2026-01-10,Bike light,Electronics,199.2,2,True,NaN,NaN


In [23]:
print(orders_raw.duplicated().sum())


20


In [24]:
print(orders.duplicated(subset='order_id').sum())

25


In [25]:
total_with = (orders['unit_price'] * orders['quantity']).sum()

orders = orders.drop_duplicates().reset_index(drop=True)

total_without = (orders['unit_price'] * orders['quantity']).sum()

print(orders.shape)
print(round(total_with, 2))
print(round(total_without, 2))
print(round(total_with - total_without, 2))


(800, 10)
591385.2
572783.8
18601.4


## Drop what we do not need: drop()

In [26]:
orders = orders.drop(columns='gift_message')

orders.shape

(800, 9)

**Which category brought in the most money?**

In [27]:
orders['total'] = orders['unit_price'] * orders['quantity']

orders.groupby('category')['total'].sum().round(0).sort_values(ascending=False)



category
Shoes            207382.0
Clothing         153808.0
Electronics       93745.0
Sports            56565.0
Home & Garden     44604.0
Books             16680.0
Name: total, dtype: float64

In [28]:
monthly = orders.groupby(orders['order_date'].dt.to_period('M'))['total'].sum()

print(monthly.round(0))
print(round(monthly.sum(), 2))
print(round(orders['total'].sum(), 2))

order_date
2026-01    106577.0
2026-02     93654.0
2026-03     91064.0
2026-04     74254.0
2026-05    126013.0
2026-06     78478.0
Freq: M, Name: total, dtype: float64
570040.8
572783.8


In [29]:
orders_clean = orders_raw.copy()

# 1. Category: remove spaces, lowercase, one spelling, capital letters
orders_clean['category'] = (
    orders_clean['category']
    .str.strip()
    .str.lower()
    .replace('home and garden', 'home & garden')
    .str.title()
)

# 2. Express: yes/no in many spellings -> True/False
orders_clean['express'] = orders_clean['express'].str.lower().map({'yes': True, 'y': True, 'no': False})

# 3. Price: text like '1 199,00 kr' -> number 1199.0
orders_clean['unit_price'] = (
    orders_clean['unit_price']
    .str.strip()
    .str.replace(' kr', '')
    .str.replace(' ', '')
    .str.replace(',', '.')
    .astype(float)
)

# 4. Date: text -> real dates. Impossible dates become missing (NaT)
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')

# 5. Duplicates: remove exact copies. This comes AFTER the text cleaning
orders_clean = orders_clean.drop_duplicates().reset_index(drop=True)

# 6. The empty column goes, and a new column with the amount for every row
orders_clean = orders_clean.drop(columns='gift_message')
orders_clean['total'] = orders_clean['unit_price'] * orders_clean['quantity']

print(orders_clean.shape)
print(orders_clean.equals(orders))

(800, 10)
True


In [30]:
orders_clean.to_csv('data/orders_clean.csv', index=False)